In [67]:
import pandas as pd

df = pd.read_csv("housing.csv")

In [68]:
#data cleaning
"""fill missing values with the median"""
df['total_bedrooms'].fillna(df['total_bedrooms'].median(), inplace=True)


"""encode categorical variables with dummies"""
locations = df['ocean_proximity'].str.get_dummies()
data = pd.concat((df, locations), axis=1)
data = data.drop('ocean_proximity', axis=1)

In [69]:
"""convert median house value to 2025 prices
limitation: can convert to 2025 prices, but may not be an accurate reflection of prices due to price gouging
1990 CPI: 130.7
2025 CPI: 319.80
adjustment: 2.45"""
def to_thousand(x):
    return x * 10000
data

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,0,0,0,1,0
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,0,0,0,1,0
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,0,0,0,1,0
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,0,0,0,1,0
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,0,1,0,0,0
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,0,1,0,0,0
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,0,1,0,0,0
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,0,1,0,0,0


In [ ]:
#convert to ten thousands to be more readable
data['median_income'] = data['median_income'].apply(to_thousand)

In [71]:
data

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,83252.0,452600.0,0,0,0,1,0
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,83014.0,358500.0,0,0,0,1,0
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,72574.0,352100.0,0,0,0,1,0
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,56431.0,341300.0,0,0,0,1,0
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,38462.0,342200.0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,15603.0,78100.0,0,1,0,0,0
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,25568.0,77100.0,0,1,0,0,0
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,17000.0,92300.0,0,1,0,0,0
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,18672.0,84700.0,0,1,0,0,0


In [72]:
"""features that will be selected:
ocean proximity 
longtiude and latitude
housing median age
total rooms
total bedrooms
population
households
median income 

target value:
median_house_value"""
X = data[['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', '<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN']]
y = data['median_house_value']

In [73]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=12)

In [74]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(16512, 13)
(4128, 13)
(16512,)
(4128,)


In [75]:
#use linear regression model to predict housing prices
from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(X_train, y_train)

LinearRegression()

In [76]:
#make predictions
y_pred = model.predict(X_test)

In [77]:
#evaluate the model
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(mse)
print(r2)

4894945981.353006
0.6430110495329134


In [78]:
import math
avg_error = math.sqrt(mse)
print(f"average error: {avg_error}")

average error: 69963.89055329189


In [79]:
#preparing for new dataframe
import numpy as np
y_test = y_test.reset_index(drop=True)
predictions_series = pd.Series(y_pred, name='predicted_value')
errors = np.abs(y_test - predictions_series)
errors.name = 'error'
X_test = X_test.reset_index(drop=True)

In [80]:
#show results
results = pd.concat([X_test, y_test.rename('actual_price'), predictions_series, errors], axis=1)
results

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN,actual_price,predicted_value,error
0,-118.13,34.04,42.0,2205.0,451.0,1392.0,423.0,43646.0,1,0,0,0,0,211400.0,244857.821113,33457.821113
1,-122.09,37.65,27.0,2630.0,722.0,1414.0,634.0,28203.0,0,0,0,1,0,195200.0,213359.196738,18159.196738
2,-117.71,33.63,16.0,2497.0,500.0,1357.0,456.0,45909.0,1,0,0,0,0,241800.0,232060.862354,9739.137646
3,-120.43,34.69,33.0,2054.0,373.0,1067.0,358.0,36023.0,0,0,0,0,1,128300.0,259153.652649,130853.652649
4,-121.53,39.06,20.0,561.0,109.0,308.0,114.0,33021.0,0,1,0,0,0,70800.0,107792.613732,36992.613732
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4123,-118.35,34.09,35.0,2234.0,689.0,1334.0,662.0,25444.0,1,0,0,0,0,236100.0,211062.420043,25037.579957
4124,-118.37,34.14,8.0,4382.0,1560.0,2138.0,1411.0,35714.0,1,0,0,0,0,197900.0,301561.132223,103661.132223
4125,-121.60,39.77,23.0,2263.0,497.0,1138.0,455.0,23403.0,0,1,0,0,0,87300.0,70642.844126,16657.155874
4126,-122.47,37.77,52.0,3143.0,635.0,1350.0,623.0,38571.0,0,0,0,1,0,366700.0,280459.476137,86240.523863


In [81]:
results.sort_values(by='error', ascending=False).head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN,actual_price,predicted_value,error
3014,-117.42,33.35,14.0,25135.0,4819.0,35682.0,4769.0,25729.0,1,0,0,0,0,134400.0,-751944.051893,886344.051893
2472,-121.29,37.80,6.0,110.0,26.0,69.0,24.0,37292.0,0,1,0,0,0,475000.0,133918.326224,341081.673776
2801,-121.90,37.39,42.0,42.0,14.0,26.0,14.0,17361.0,1,0,0,0,0,500001.0,161932.359431,338068.640569
13,-118.31,34.06,36.0,369.0,147.0,145.0,136.0,8804.0,1,0,0,0,0,450000.0,124473.150446,325526.849554
3614,-119.14,34.23,8.0,243.0,75.0,102.0,80.0,25714.0,0,0,0,0,1,500001.0,175392.683276,324608.316724


In [83]:
results.sort_values(by='error', ascending=True).head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,<1H OCEAN,INLAND,ISLAND,NEAR BAY,NEAR OCEAN,actual_price,predicted_value,error
919,-121.29,37.96,52.0,287.0,119.0,154.0,85.0,8738.0,0,1,0,0,0,75000.0,75007.202288,7.202288
3798,-117.92,33.94,27.0,4566.0,620.0,2045.0,664.0,55830.0,1,0,0,0,0,267700.0,267684.340854,15.659146
3546,-117.89,34.07,32.0,2374.0,450.0,1580.0,427.0,38837.0,1,0,0,0,0,200300.0,200246.056900,53.943100
1653,-121.88,37.32,30.0,1242.0,338.0,1438.0,325.0,26607.0,1,0,0,0,0,169300.0,169371.179410,71.179410
1663,-120.93,38.50,15.0,1248.0,234.0,529.0,216.0,33393.0,0,1,0,0,0,107200.0,107107.298127,92.701873
